# Resultados Consolidados - Avaliação de RAG para Nutrição

Este notebook gera todas as tabelas e figuras necessárias para o capítulo de Resultados do TCC.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path

matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['savefig.dpi'] = 300
matplotlib.rcParams['font.size'] = 11
matplotlib.rcParams['axes.titlesize'] = 13
matplotlib.rcParams['axes.labelsize'] = 11

RESULTS_DIR = Path('../results')
PLOTS_DIR = RESULTS_DIR / 'plots'
PLOTS_DIR.mkdir(exist_ok=True)

In [ ]:
# Load data
ragas = pd.read_csv(RESULTS_DIR / 'ragas_benchmark_results.csv')
deterministic = pd.read_csv(RESULTS_DIR / 'deterministic_benchmark_results.csv')
ragas_stats = pd.read_csv(RESULTS_DIR / 'ragas_statistical_summary.csv')
det_stats = pd.read_csv(RESULTS_DIR / 'deterministic_statistical_summary.csv')
correlation = pd.read_csv(RESULTS_DIR / 'correlation_analysis.csv')
no_rag = pd.read_csv(RESULTS_DIR / 'no_rag_comparison.csv')
failure = pd.read_csv(RESULTS_DIR / 'failure_analysis.csv')

---
## Tabela 1 - Configurações Experimentais

In [ ]:
# Table 1 - Experimental Configurations
configs = {
    'ID': ['E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'E10', 'E11', 'E12', 'E13', 'E14'],
    'Configuração': [
        'no\_rag\_gpt35', 'no\_rag\_llama8b',
        'fair\_gpt35', 'fair\_llama8b',
        'llama\_naive', 'llama\_cs256', 'llama\_cs1024',
        'llama\_ov0', 'llama\_ov100',
        'llama\_topk3', 'llama\_topk8',
        'llama\_reranker', 'llama\_hyde', 'llama\_temp030'
    ],
    'LLM': [
        'GPT-3.5 Turbo', 'Llama 3.1 8B',
        'GPT-3.5 Turbo', 'Llama 3.1 8B',
        'Llama 3.1 8B', 'Llama 3.1 8B', 'Llama 3.1 8B',
        'Llama 3.1 8B', 'Llama 3.1 8B',
        'Llama 3.1 8B', 'Llama 3.1 8B',
        'Llama 3.1 8B', 'Llama 3.1 8B', 'Llama 3.1 8B'
    ],
    'RAG': [
        'Não', 'Não',
        'Sim', 'Sim',
        'Sim', 'Sim', 'Sim',
        'Sim', 'Sim',
        'Sim', 'Sim',
        'Sim', 'Sim', 'Sim'
    ],
    'Chunk Size': [
        '-', '-',
        '512', '512',
        '512', '256', '1024',
        '512', '512',
        '512', '512',
        '512', '512', '512'
    ],
    'Overlap': [
        '-', '-',
        '50', '50',
        '50', '50', '50',
        '0', '100',
        '50', '50',
        '50', '50', '50'
    ],
    'Top-K': [
        '-', '-',
        '5', '5',
        '5', '5', '5',
        '5', '5',
        '3', '8',
        '5', '5', '5'
    ],
    'Reranker': [
        '-', '-',
        'Não', 'Não',
        'Não', 'Não', 'Não',
        'Não', 'Não',
        'Não', 'Não',
        'Sim', 'Não (HyDE)', 'Não'
    ],
    'Temperatura': [
        '0.0', '0.0',
        '0.0', '0.0',
        '0.0', '0.0', '0.0',
        '0.0', '0.0',
        '0.0', '0.0',
        '0.0', '0.0', '0.3'
    ]
}

df_configs = pd.DataFrame(configs)
print(df_configs.to_latex(index=False, escape=False))

---
## Tabela 2 - Resultados Completos (RAGAS + Determinísticos)

Consolida todas as métricas RAGAS (faithfulness, answer relevancy, context precision, context recall, answer correctness) e determinísticas (BLEU, ROUGE-L, METEOR, BERTScore F1, Semantic Similarity).

In [ ]:
# Merge RAGAS + Deterministic results
ragas_metrics = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall', 'answer_correctness']
det_metrics = ['bleu', 'rouge_l', 'meteor', 'bertscore_f1', 'semantic_similarity']

merged = ragas[['config'] + ragas_metrics + ['avg_latency_ms']].merge(
    deterministic[['config'] + det_metrics],
    on='config'
)

# Compute a composite "Final Score" as the mean of all 10 metrics
all_metrics = ragas_metrics + det_metrics
merged['score_final'] = merged[all_metrics].mean(axis=1)

# Sort by final score descending
merged = merged.sort_values('score_final', ascending=False).reset_index(drop=True)

# Display with nice formatting
display_cols = ['config'] + ragas_metrics + det_metrics + ['score_final', 'avg_latency_ms']
display_df = merged[display_cols].copy()
for col in ragas_metrics + det_metrics + ['score_final']:
    display_df[col] = display_df[col].map(lambda x: f'{x:.4f}')
display_df['avg_latency_ms'] = display_df['avg_latency_ms'].map(lambda x: f'{x/1000:.1f}s')
display_df = display_df.rename(columns={'avg_latency_ms': 'Latência', 'score_final': 'Score Final'})

print('=== TABELA 2 - Resultados Completos ===')
display_df

In [ ]:
# LaTeX table for the thesis
latex_df = merged[display_cols].copy()
for col in ragas_metrics + det_metrics + ['score_final']:
    latex_df[col] = latex_df[col].map(lambda x: f'{x:.4f}')
latex_df['avg_latency_ms'] = latex_df['avg_latency_ms'].map(lambda x: f'{x/1000:.1f}')

latex_df = latex_df.rename(columns={
    'config': 'Configuração',
    'faithfulness': 'Faith.',
    'answer_relevancy': 'Ans. Rel.',
    'context_precision': 'Ctx. Prec.',
    'context_recall': 'Ctx. Recall',
    'answer_correctness': 'Ans. Corr.',
    'bleu': 'BLEU',
    'rouge_l': 'ROUGE-L',
    'meteor': 'METEOR',
    'bertscore_f1': 'BERTScore',
    'semantic_similarity': 'Sem. Sim.',
    'score_final': 'Score',
    'avg_latency_ms': 'Lat.(s)'
})

print(latex_df.to_latex(index=True, escape=False, float_format='%.4f'))

---
## Tabela 3 - Ranking de Configurações

In [ ]:
# Table 3 - Ranking
ranking = merged[['config', 'score_final']].copy()
ranking['Posição'] = range(1, len(ranking) + 1)
ranking = ranking[['Posição', 'config', 'score_final']]
ranking.columns = ['Posição', 'Configuração', 'Score Final']
ranking['Score Final'] = ranking['Score Final'].map(lambda x: f'{x:.4f}')

print('=== TABELA 3 - Ranking de Configurações ===')
print(ranking.to_latex(index=False, escape=False))
ranking

---
## Figura 1 - Comparação Geral (Score Final)

In [ ]:
# Figure 1 - Horizontal bar chart of Final Scores
plot_data = merged[['config', 'score_final']].sort_values('score_final')

fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(plot_data)))
bars = ax.barh(plot_data['config'], plot_data['score_final'], color=colors, edgecolor='gray', linewidth=0.5)

for bar, val in zip(bars, plot_data['score_final']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.4f}',
            va='center', ha='left', fontsize=9)

ax.set_xlabel('Score Final (média de todas as métricas)')
ax.set_title('Figura 1 - Comparação Geral das Configurações')
ax.set_xlim(0, max(plot_data['score_final']) * 1.12)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig1_overall_comparison.png', bbox_inches='tight')
plt.show()

---
## Figura 2 - Métricas por Configuração (Barras Agrupadas)

In [ ]:
# Figure 2 - Grouped bars for RAGAS metrics
# Focus on RAGAS metrics only (most important for RAG evaluation)
ragas_plot = merged[['config'] + ragas_metrics].sort_values('faithfulness', ascending=False)

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(ragas_plot))
width = 0.15
colors_ragas = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

for i, (metric, color) in enumerate(zip(ragas_metrics, colors_ragas)):
    ax.bar(x + i * width, ragas_plot[metric], width, label=metric.replace('_', ' ').title(),
           color=color, edgecolor='gray', linewidth=0.3)

ax.set_xlabel('Configuração')
ax.set_ylabel('Score')
ax.set_title('Figura 2 - Métricas RAGAS por Configuração')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(ragas_plot['config'], rotation=45, ha='right', fontsize=9)
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig2_ragas_by_config.png', bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2b - Grouped bars for Deterministic metrics
det_plot = merged[['config'] + det_metrics].copy()
# Use normalized metrics for visualization (BLEU, ROUGE-L, METEOR are on different scales)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left panel: BLEU, ROUGE-L, METEOR (lexical metrics)
lexical = ['bleu', 'rouge_l', 'meteor']
det_sorted = det_plot.sort_values('meteor', ascending=False)
x = np.arange(len(det_sorted))
width = 0.25
colors_lex = ['#2196F3', '#4CAF50', '#FF9800']

for i, (metric, color) in enumerate(zip(lexical, colors_lex)):
    axes[0].bar(x + i * width, det_sorted[metric], width, label=metric.upper(),
                color=color, edgecolor='gray', linewidth=0.3)

axes[0].set_xlabel('Configuração')
axes[0].set_ylabel('Score')
axes[0].set_title('Métricas Léxicas')
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(det_sorted['config'], rotation=45, ha='right', fontsize=8)
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)

# Right panel: BERTScore, Semantic Similarity (semantic metrics)
semantic = ['bertscore_f1', 'semantic_similarity']
colors_sem = ['#9C27B0', '#F44336']

for i, (metric, color) in enumerate(zip(semantic, colors_sem)):
    axes[1].bar(x + i * width, det_sorted[metric], width, label=metric.replace('_', ' ').title(),
                color=color, edgecolor='gray', linewidth=0.3)

axes[1].set_xlabel('Configuração')
axes[1].set_ylabel('Score')
axes[1].set_title('Métricas Semânticas')
axes[1].set_xticks(x + width / 2)
axes[1].set_xticklabels(det_sorted['config'], rotation=45, ha='right', fontsize=8)
axes[1].legend(fontsize=8)
axes[1].set_ylim(0.85, 1.0)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Figura 2b - Métricas Determinísticas por Configuração', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig2b_deterministic_by_config.png', bbox_inches='tight')
plt.show()

---
## Tabela 4 - Melhor Resultado por Métrica

In [ ]:
# Table 4 - Best result per metric
best_per_metric = []
for metric in all_metrics:
    best_idx = merged[metric].idxmax()
    best_per_metric.append({
        'Métrica': metric.replace('_', ' ').title(),
        'Melhor Configuração': merged.loc[best_idx, 'config'],
        'Valor': f"{merged.loc[best_idx, metric]:.4f}"
    })

df_best = pd.DataFrame(best_per_metric)
print('=== TABELA 4 - Melhor Resultado por Métrica ===')
print(df_best.to_latex(index=False, escape=False))
df_best

---
## Figura 3 - Impacto do Chunk Size

In [ ]:
# Figure 3 - Impact of chunk size
chunk_configs = ['llama_cs256', 'llama_naive', 'llama_cs1024']  # 256, 512(default), 1024
chunk_sizes = [256, 512, 1024]
chunk_data = merged[merged['config'].isin(chunk_configs)].copy()
chunk_data['chunk_size'] = chunk_data['config'].map(dict(zip(chunk_configs, chunk_sizes)))
chunk_data = chunk_data.sort_values('chunk_size')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(chunk_data['chunk_size'], chunk_data['faithfulness'], 'o-', label='Faithfulness', linewidth=2, markersize=8)
ax.plot(chunk_data['chunk_size'], chunk_data['answer_relevancy'], 's-', label='Answer Relevancy', linewidth=2, markersize=8)
ax.plot(chunk_data['chunk_size'], chunk_data['context_precision'], '^-', label='Context Precision', linewidth=2, markersize=8)
ax.plot(chunk_data['chunk_size'], chunk_data['context_recall'], 'D-', label='Context Recall', linewidth=2, markersize=8)
ax.plot(chunk_data['chunk_size'], chunk_data['score_final'], 'v-', label='Score Final', linewidth=2.5, markersize=10, color='black')

ax.set_xlabel('Chunk Size')
ax.set_ylabel('Score')
ax.set_title('Figura 3 - Impacto do Chunk Size nas Métricas')
ax.set_xticks(chunk_sizes)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax.set_ylim(0.3, 1.0)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig3_chunk_size_impact.png', bbox_inches='tight')
plt.show()

---
## Figura 4 - Impacto do Reranking

In [ ]:
# Figure 4 - Impact of reranking
# llama_naive = no reranker, llama_reranker = with reranker (same base config otherwise)
rerank_configs = ['llama_naive', 'llama_reranker']
rerank_labels = ['Sem Reranker', 'Com Reranker']
rerank_data = merged[merged['config'].isin(rerank_configs)].copy()
rerank_data['label'] = rerank_data['config'].map(dict(zip(rerank_configs, rerank_labels)))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# RAGAS metrics comparison
ragas_vals = rerank_data.set_index('label')[ragas_metrics].T
ragas_vals.plot(kind='bar', ax=axes[0], color=['#2196F3', '#FF9800'], edgecolor='gray', linewidth=0.5)
axes[0].set_title('Métricas RAGAS')
axes[0].set_ylabel('Score')
axes[0].set_xticklabels([m.replace('_', ' ').title() for m in ragas_metrics], rotation=30, ha='right', fontsize=9)
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim(0, 1.05)

# Final score comparison
final_vals = rerank_data.set_index('label')['score_final']
bars = axes[1].bar(final_vals.index, final_vals.values, color=['#2196F3', '#FF9800'], edgecolor='gray', linewidth=0.5)
for bar, val in zip(bars, final_vals.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.005, f'{val:.4f}',
                ha='center', va='bottom', fontsize=10)
axes[1].set_title('Score Final')
axes[1].set_ylabel('Score')
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim(0, max(final_vals.values) * 1.1)

plt.suptitle('Figura 4 - Impacto do Reranking', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig4_reranker_impact.png', bbox_inches='tight')
plt.show()

---
## Figura 5 - Impacto do HyDE

In [ ]:
# Figure 5 - HyDE comparison
hyde_configs = ['llama_naive', 'llama_hyde']
hyde_labels = ['Sem HyDE', 'Com HyDE']
hyde_data = merged[merged['config'].isin(hyde_configs)].copy()
hyde_data['label'] = hyde_data['config'].map(dict(zip(hyde_configs, hyde_labels)))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ragas_vals = hyde_data.set_index('label')[ragas_metrics].T
ragas_vals.plot(kind='bar', ax=axes[0], color=['#2196F3', '#4CAF50'], edgecolor='gray', linewidth=0.5)
axes[0].set_title('Métricas RAGAS')
axes[0].set_ylabel('Score')
axes[0].set_xticklabels([m.replace('_', ' ').title() for m in ragas_metrics], rotation=30, ha='right', fontsize=9)
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim(0, 1.05)

final_vals = hyde_data.set_index('label')['score_final']
bars = axes[1].bar(final_vals.index, final_vals.values, color=['#2196F3', '#4CAF50'], edgecolor='gray', linewidth=0.5)
for bar, val in zip(bars, final_vals.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.005, f'{val:.4f}',
                ha='center', va='bottom', fontsize=10)
axes[1].set_title('Score Final')
axes[1].set_ylabel('Score')
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim(0, max(final_vals.values) * 1.1)

plt.suptitle('Figura 5 - Impacto do HyDE', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig5_hyde_impact.png', bbox_inches='tight')
plt.show()

---
## Figura 6 - Comparação RAG vs No-RAG

In [ ]:
# Figure 6 - RAG vs No-RAG comparison
# Fair comparison configs only (same LLM)
rag_vs_norag = {
    'GPT-3.5 RAG': merged[merged['config'] == 'fair_gpt35_turbo'][ragas_metrics].values[0],
    'GPT-3.5 No-RAG': merged[merged['config'] == 'no_rag_gpt35_turbo'][ragas_metrics].values[0],
    'Llama 8B RAG': merged[merged['config'] == 'fair_llama8b'][ragas_metrics].values[0],
    'Llama 8B No-RAG': merged[merged['config'] == 'no_rag_llama8b'][ragas_metrics].values[0],
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# GPT comparison
x = np.arange(len(ragas_metrics))
width = 0.35
axes[0].bar(x - width/2, rag_vs_norag['GPT-3.5 RAG'], width, label='Com RAG', color='#4CAF50')
axes[0].bar(x + width/2, rag_vs_norag['GPT-3.5 No-RAG'], width, label='Sem RAG', color='#F44336')
axes[0].set_title('GPT-3.5 Turbo')
axes[0].set_xticks(x)
axes[0].set_xticklabels([m.replace('_', ' ').title() for m in ragas_metrics], rotation=30, ha='right', fontsize=9)
axes[0].legend()
axes[0].set_ylim(0, 1.05)
axes[0].grid(axis='y', alpha=0.3)

# Llama comparison
axes[1].bar(x - width/2, rag_vs_norag['Llama 8B RAG'], width, label='Com RAG', color='#4CAF50')
axes[1].bar(x + width/2, rag_vs_norag['Llama 8B No-RAG'], width, label='Sem RAG', color='#F44336')
axes[1].set_title('Llama 3.1 8B')
axes[1].set_xticks(x)
axes[1].set_xticklabels([m.replace('_', ' ').title() for m in ragas_metrics], rotation=30, ha='right', fontsize=9)
axes[1].legend()
axes[1].set_ylim(0, 1.05)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Figura 6 - Comparação RAG vs No-RAG (Métricas RAGAS)', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig6_rag_vs_norag.png', bbox_inches='tight')
plt.show()

---
## Figura 7 - Impacto da Temperatura

In [ ]:
# Figure 7 - Temperature impact (0.0 vs 0.3)
temp_configs = ['llama_naive', 'llama_temp030']
temp_labels = ['Temp. 0.0', 'Temp. 0.3']
temp_data = merged[merged['config'].isin(temp_configs)].copy()
temp_data['label'] = temp_data['config'].map(dict(zip(temp_configs, temp_labels)))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(all_metrics))
width = 0.35

temp_vals = temp_data.set_index('label')[all_metrics]
ax.bar(x - width/2, temp_vals.loc['Temp. 0.0'], width, label='Temperatura 0.0', color='#2196F3')
ax.bar(x + width/2, temp_vals.loc['Temp. 0.3'], width, label='Temperatura 0.3', color='#FF9800')

ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', ' ').title() for m in all_metrics], rotation=40, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_title('Figura 7 - Impacto da Temperatura (0.0 vs 0.3)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig7_temperature_impact.png', bbox_inches='tight')
plt.show()

---
## Figura 8 - Impacto do Top-K

In [ ]:
# Figure 8 - Top-K impact
topk_configs = ['llama_topk3', 'llama_naive', 'llama_topk8']  # top_k 3, 5(default), 8
topk_values = [3, 5, 8]
topk_data = merged[merged['config'].isin(topk_configs)].copy()
topk_data['topk'] = topk_data['config'].map(dict(zip(topk_configs, topk_values)))
topk_data = topk_data.sort_values('topk')

fig, ax = plt.subplots(figsize=(8, 5))
for metric, color, marker in zip(ragas_metrics,
                                   ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336'],
                                   ['o', 's', '^', 'D', 'v']):
    ax.plot(topk_data['topk'], topk_data[metric], f'{marker}-', label=metric.replace('_', ' ').title(),
            linewidth=2, markersize=8, color=color)

ax.plot(topk_data['topk'], topk_data['score_final'], 'k*-', label='Score Final', linewidth=2.5, markersize=12)

ax.set_xlabel('Top-K')
ax.set_ylabel('Score')
ax.set_title('Figura 8 - Impacto do Top-K')
ax.set_xticks(topk_values)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig8_topk_impact.png', bbox_inches='tight')
plt.show()

---
## Figura 9 - Impacto do Overlap

In [ ]:
# Figure 9 - Overlap impact (0, 50, 100)
ov_configs = ['llama_ov0', 'llama_naive', 'llama_ov100']
ov_values = [0, 50, 100]
ov_data = merged[merged['config'].isin(ov_configs)].copy()
ov_data['overlap'] = ov_data['config'].map(dict(zip(ov_configs, ov_values)))
ov_data = ov_data.sort_values('overlap')

fig, ax = plt.subplots(figsize=(8, 5))
for metric, color, marker in zip(ragas_metrics,
                                   ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336'],
                                   ['o', 's', '^', 'D', 'v']):
    ax.plot(ov_data['overlap'], ov_data[metric], f'{marker}-', label=metric.replace('_', ' ').title(),
            linewidth=2, markersize=8, color=color)

ax.plot(ov_data['overlap'], ov_data['score_final'], 'k*-', label='Score Final', linewidth=2.5, markersize=12)

ax.set_xlabel('Overlap')
ax.set_ylabel('Score')
ax.set_title('Figura 9 - Impacto do Overlap')
ax.set_xticks(ov_values)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig9_overlap_impact.png', bbox_inches='tight')
plt.show()

---
## Figura 10 - Heatmap de Correlação RAGAS vs Determinísticas

In [ ]:
# Figure 10 - Correlation heatmap
corr_pivot = correlation.pivot(index='ragas_metric', columns='deterministic_metric', values='pearson_r')
corr_pivot.index = [m.replace('_', ' ').title() for m in corr_pivot.index]
corr_pivot.columns = [m.replace('_', ' ').title() for m in corr_pivot.columns]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_pivot.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(corr_pivot.columns)))
ax.set_yticks(range(len(corr_pivot.index)))
ax.set_xticklabels(corr_pivot.columns, rotation=30, ha='right', fontsize=9)
ax.set_yticklabels(corr_pivot.index, fontsize=9)

for i in range(len(corr_pivot.index)):
    for j in range(len(corr_pivot.columns)):
        val = corr_pivot.values[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9, color=color)

plt.colorbar(im, ax=ax, label='Pearson r')
ax.set_title('Figura 10 - Correlação Pearson: Métricas RAGAS vs Determinísticas')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig10_correlation_heatmap.png', bbox_inches='tight')
plt.show()

---
## Tabela 5 - Exemplos de Falha

Seleciona 5 exemplos representativos de falhas do melhor modelo (llama_hyde).

In [ ]:
# Table 5 - Failure examples from the best configuration
# Get failures from llama_hyde (best config)
best_config_failures = failure[failure['config'] == 'llama_hyde'].copy()

# Filter for actual failures (score_final < 0.3 or multiple failure types)
best_config_failures['failure_count'] = best_config_failures['failure_types'].apply(
    lambda x: len(eval(x)) if isinstance(x, str) and x.startswith('[') else 0
)
best_config_failures['avg_ragas'] = best_config_failures[ragas_metrics].mean(axis=1)

# Get worst 5 by average RAGAS score
worst_5 = best_config_failures.nsmallest(5, 'avg_ragas')

print('=== TABELA 5 - Exemplos de Falha (llama\_hyde) ===')
for idx, row in worst_5.iterrows():
    print(f"\nPergunta: {row['question']}")
    print(f"Tipos de falha: {row['failure_types']}")
    print(f"  Faithfulness: {row['faithfulness']:.2f} | Ans. Rel.: {row['answer_relevancy']:.2f} | "
          f"Ctx. Prec.: {row['context_precision']:.2f} | Ctx. Recall: {row['context_recall']:.2f} | "
          f"Ans. Corr.: {row['answer_correctness']:.2f}")
    print(f"  Sem. Sim.: {row['semantic_similarity']:.4f} | BERTScore: {row['bertscore_f1']:.4f}")

---
## Tabela 6 - Resumo Estatístico (Média ± Desvio Padrão)

In [ ]:
# Table 6 - Statistical summary (mean +/- std)
ragas_stats_pivot = ragas_stats.pivot(index='config', columns='metric', values=['mean', 'std'])
det_stats_pivot = det_stats.pivot(index='config', columns='metric', values=['mean', 'std'])

# Build summary table
summary_rows = []
for config in ragas_stats['config'].unique():
    row = {'Config': config}
    for metric in ragas_metrics:
        if metric in ragas_stats_pivot['mean'].columns:
            mean_val = ragas_stats_pivot['mean'].loc[config, metric]
            std_val = ragas_stats_pivot['std'].loc[config, metric]
            row[metric] = f'{mean_val:.4f} ± {std_val:.4f}'
    for metric in det_metrics:
        if config in det_stats_pivot['mean'].index and metric in det_stats_pivot['mean'].columns:
            mean_val = det_stats_pivot['mean'].loc[config, metric]
            std_val = det_stats_pivot['std'].loc[config, metric]
            row[metric] = f'{mean_val:.4f} ± {std_val:.4f}'
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
# Sort by config order from merged
config_order = merged['config'].tolist()
df_summary['sort_key'] = df_summary['Config'].map({c: i for i, c in enumerate(config_order)})
df_summary = df_summary.sort_values('sort_key').drop('sort_key', axis=1).reset_index(drop=True)

print('=== TABELA 6 - Resumo Estatístico (Média ± Desvio Padrão) ===')
df_summary

---
## Figura 11 - Comparação de Latência

In [ ]:
# Figure 11 - Latency comparison
latency_data = merged[['config', 'avg_latency_ms', 'score_final']].sort_values('avg_latency_ms')

fig, ax1 = plt.subplots(figsize=(12, 6))

x = np.arange(len(latency_data))
bars = ax1.bar(x, latency_data['avg_latency_ms'] / 1000, color='#2196F3', alpha=0.7, label='Latência (s)')
ax1.set_ylabel('Latência (segundos)', color='#2196F3')
ax1.tick_params(axis='y', labelcolor='#2196F3')

ax2 = ax1.twinx()
ax2.plot(x, latency_data['score_final'], 'r*-', linewidth=2, markersize=12, label='Score Final')
ax2.set_ylabel('Score Final', color='#F44336')
ax2.tick_params(axis='y', labelcolor='#F44336')

ax1.set_xticks(x)
ax1.set_xticklabels(latency_data['config'], rotation=45, ha='right', fontsize=9)
ax1.set_title('Figura 11 - Latência vs Score Final')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

ax1.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fig11_latency_vs_score.png', bbox_inches='tight')
plt.show()

---
## Resumo Final - Dados Consolidados em JSON

In [ ]:
import json

# Build consolidated JSON with all results
consolidated = {
    'experiments': []
}

config_info = {
    'no_rag_gpt35_turbo': {'llm': 'GPT-3.5 Turbo', 'rag': False, 'chunk_size': None, 'overlap': None, 'top_k': None, 'reranker': False, 'hyde': False, 'temperature': 0.0},
    'no_rag_llama8b': {'llm': 'Llama 3.1 8B', 'rag': False, 'chunk_size': None, 'overlap': None, 'top_k': None, 'reranker': False, 'hyde': False, 'temperature': 0.0},
    'fair_gpt35_turbo': {'llm': 'GPT-3.5 Turbo', 'rag': True, 'chunk_size': 512, 'overlap': 50, 'top_k': 5, 'reranker': False, 'hyde': False, 'temperature': 0.0},
    'fair_llama8b': {'llm': 'Llama 3.1 8B', 'rag': True, 'chunk_size': 512, 'overlap': 50, 'top_k': 5, 'reranker': False, 'hyde': False, 'temperature': 0.0},
    'llama_naive': {'llm': 'Llama 3.1 8B', 'rag': True, 'chunk_size': 512, 'overlap': 50, 'top_k': 5, 'reranker': False, 'hyde': False, 'temperature': 0.0},
    'llama_cs256': {'llm': 'Llama 3.1 8B', 'rag': True, 'chunk_size': 256, 'overlap': 50, 'top_k': 5, 'reranker': False, 'hyde': False, 'temperature': 0.0},
    'llama_cs1024': {'llm': 'Llama 3.1 8B', 'rag': True, 'chunk_size': 1024, 'overlap': 50, 'top_k': 5, 'reranker': False, 'hyde': False, 'temperature': 0.0},
    'llama_ov0': {'llm': 'Llama 3.1 8B', 'rag': True, 'chunk_size': 512, 'overlap': 0, 'top_k': 5, 'reranker': False, 'hyde': False, 'temperature': 0.0},
    'llama_ov100': {'llm': 'Llama 3.1 8B', 'rag': True, 'chunk_size': 512, 'overlap': 100, 'top_k': 5, 'reranker': False, 'hyde': False, 'temperature': 0.0},
    'llama_topk3': {'llm': 'Llama 3.1 8B', 'rag': True, 'chunk_size': 512, 'overlap': 50, 'top_k': 3, 'reranker': False, 'hyde': False, 'temperature': 0.0},
    'llama_topk8': {'llm': 'Llama 3.1 8B', 'rag': True, 'chunk_size': 512, 'overlap': 50, 'top_k': 8, 'reranker': False, 'hyde': False, 'temperature': 0.0},
    'llama_reranker': {'llm': 'Llama 3.1 8B', 'rag': True, 'chunk_size': 512, 'overlap': 50, 'top_k': 5, 'reranker': True, 'hyde': False, 'temperature': 0.0},
    'llama_hyde': {'llm': 'Llama 3.1 8B', 'rag': True, 'chunk_size': 512, 'overlap': 50, 'top_k': 5, 'reranker': False, 'hyde': True, 'temperature': 0.0},
    'llama_temp030': {'llm': 'Llama 3.1 8B', 'rag': True, 'chunk_size': 512, 'overlap': 50, 'top_k': 5, 'reranker': False, 'hyde': False, 'temperature': 0.3},
}

for _, row in merged.iterrows():
    config = row['config']
    info = config_info[config]
    experiment = {
        'id': config,
        'configuration': info,
        'ragas_metrics': {m: round(row[m], 6) for m in ragas_metrics},
        'deterministic_metrics': {m: round(row[m], 6) for m in det_metrics},
        'score_final': round(row['score_final'], 6),
        'latency_seconds': round(row['avg_latency_ms'] / 1000, 2)
    }
    consolidated['experiments'].append(experiment)

# Save JSON
json_path = RESULTS_DIR / 'consolidated_results.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(consolidated, f, ensure_ascii=False, indent=2)

print(f'JSON saved to {json_path}')
print(f'Total experiments: {len(consolidated["experiments"])}')
print('\nTop 5 by final score:')
for i, exp in enumerate(consolidated['experiments'][:5]):
    print(f"  {i+1}. {exp['id']}: {exp['score_final']:.4f}")

In [ ]:
# Display the full consolidated JSON
print(json.dumps(consolidated, ensure_ascii=False, indent=2))